GIL 确实限制了 Python 多线程在纯计算领域的发挥，但并没有抹杀它在**并发 I/O** 和系统架构设计中的巨大价值


| 对比维度 | GIL (Global Interpreter Lock) 全局解释器锁 | Lock / RLock |
| :--- | :--- | :--- |
| **核心本质** | 解释器级别的强制限制 | 代码级别的逻辑约定 |
| **谁来控制** | Python 解释器：解释器自动加锁、遇到 IO 或 超时自动释放。 | 程序员：需要写代码 `lock.acquire()` 和 `lock.release()`。 |
| **保护对象** | 保护“解释器不崩”维护引用计数、垃圾回收等底层结构的线程安全。 | 保护“数据不算错”，维护用户定义的变量、操作顺序等业务逻辑的完整性。 |
| **作用范围** | 全局唯一：一个进程里只有一个 GIL，管着所有线程。 | 按需创建：可以创建 100 把不同的锁，分别保护 100 个不同的业务逻辑。 |
| **如果没有** | Python 解释器可能会崩溃。 | 售票场景下，两个人可能买到同一张车票。 |



1. Node.js 天生就是为高并发 I/O 设计的（单线程 + 异步回调）。
2. Python 多线程 是受限于 GIL，为了不浪费 CPU 而被迫选择的折中方案（线程 + 释放 GIL）。
3. 在纯 I/O 高并发场景下，Python 的协程 (asyncio) 才是 Node.js 最直接的竞争对手，而 Python 的多线程更像是传统的 Java/C++ 的多线程模型。

通过并发提高 I/O 吞吐量”这个目标上，Python 的多线程和 Node.js 的理念是高度重合的。 只要任务不是纯 CPU 运算，这两者都能把 CPU 的利用率压榨到极致。

# CPU密集型演示

对比 “单线程” 和 “多线程” 跑同一段耗时的计算代码。

因为 GIL 的存在，会发现多线程跑CPU密集型任务，不仅没有变快，反而可能比单线程还要慢（因为线程切换也有开销）。

In [17]:
import time
from concurrent.futures import ThreadPoolExecutor


def cpu_intensive_task(n: int):
    count = 0
    for _ in range(n):
        count += 1
    return count


def single_thread_demo(times: int):
    """单线程执行"""
    start_time = time.perf_counter()
    for _ in range(times):
        cpu_intensive_task(50_000_000)
    end_time = time.perf_counter()
    print(f"单线程执行 {times} 次任务耗时: {end_time - start_time:.2f} 秒")


def multi_thread_demo(max_workers: int, times: int):
    start_time = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(cpu_intensive_task, 50_000_000) for _ in range(times)
        ]

        # 等待所有任务完成
        for f in futures:
            f.result()
    end_time = time.perf_counter()
    print(
        f"多线程({max_workers}个线程)执行 {times} 次任务耗时: {end_time - start_time:.2f} 秒"
    )

In [18]:
import os

print(f"当前机器 CPU 核心数: {os.cpu_count()}")
print("-" * 40)

single_thread_demo(1)
print("-" * 40)
single_thread_demo(10)
multi_thread_demo(10, 10)

当前机器 CPU 核心数: 12
----------------------------------------
单线程执行 1 次任务耗时: 2.19 秒
----------------------------------------
单线程执行 10 次任务耗时: 19.77 秒
多线程(10个线程)执行 10 次任务耗时: 21.42 秒


- **单次基准**：跑 1 次任务耗时 **2.03 秒**。
- **单线程跑 10 次**：`2.03 * 10 = 20.3`，实际耗时 **20.14 秒**。完全符合预期，因为就是排队执行。
- **多线程跑 10 次**：开了 10 个线程同时跑。你的电脑有 12 个核，如果是 C++ 或 Java，10 个核同时发力，**理论耗时应接近 2 秒**(在下面的Java测试中，居然才0.03秒)。但因为 GIL 的存在，实际耗时是 **20.36 秒**。

**结论**：多线程不仅没有变快，反而比单线程慢了 0.22 秒。这多出来的 0.22 秒，就是 10 个线程疯狂争抢 GIL 造成的**上下文切换（Context Switch）开销**。

下面的一段Java代码：运行结果

> 多线程(10个线程)执行 10 次任务耗时: **0.03 秒** (计算结果总和: 500000000)

```java
// 改成 long，防止溢出，并返回结果
long cpuIntensiveTask(int n) {
    long count = 0;
    for (int i = 0; i < n; i++) {
        count++;
    }
    return count; // 强制返回，防止被优化掉
}

void multiThreadDemo(int maxWorkers, int times) {
    long startTime = System.nanoTime();
    long totalResult = 0;
    
    try (ExecutorService executor = Executors.newFixedThreadPool(maxWorkers)) {
        List<Future<Long>> futures = new ArrayList<>();
        
        for (int i = 0; i < times; i++) {
            futures.add(executor.submit(() -> cpuIntensiveTask(50_000_000)));
        }
        
        for (Future<Long> f : futures) {
            try {
                totalResult += f.get(); // 获取返回值，强制它必须算完
            } catch (Exception e) {
                e.printStackTrace();
            }
        }
    }
    
    long endTime = System.nanoTime();
    double durationInSeconds = (endTime - startTime) / 1_000_000_000.0;
    
    System.out.printf("多线程(%d个线程)执行 %d 次任务耗时: %.2f 秒 (计算结果总和: %d)%n", maxWorkers, times, durationInSeconds, totalResult);
}

void main() {
    multiThreadDemo(10, 10);
}
```

# 数据

虽然有GIL的存在，但是数据层面的多线程竞争业务数据，仍然还是有问题的，需要用锁保护

In [ ]:
from threading import current_thread

from loguru import logger

current = 1


def sale():
    global current
    while True:
        if current <= 20:
            logger.info(f"{current_thread().name}出售了第{current}张票")
            current += 1
        else:
            logger.info("票已经售空")
            break


In [20]:
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(sale) for _ in range(20)]
    list(futures)

2026-10-23 18:01:03.917 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_0出售了第1张票
2026-10-23 18:01:03.921 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_0出售了第2张票
2026-10-23 18:01:03.921 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_1出售了第1张票
2026-10-23 18:01:03.922 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_2出售了第2张票
2026-10-23 18:01:03.923 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_3出售了第2张票
2026-10-23 18:01:03.923 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_0出售了第3张票
2026-10-23 18:01:03.926 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_4出售了第3张票
2026-10-23 18:01:03.927 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_1出售了第4张票
2026-10-23 18:01:03.932 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_2出售了第5张票
2026-10-23 18:01:03.936 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_3出售了第6张票
2026-10-23 18:01:03.937 | INFO     | __main__:sale:12 - ThreadPoolExecutor-11_0出售了第7张票
2026-10-23 18:01:03.941 | INFO     | __main

In [21]:
from threading import RLock

current = 1
lock = RLock()


def sale_save(lock):
    global current
    while True:
        with lock:
            if current <= 20:
                logger.info(f"{current_thread().name}出售了第{current}张票")
                current += 1
            else:
                logger.info("票已经售空")
                break

In [22]:
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(sale_save, lock) for _ in range(20)]
    list(futures)

2026-10-23 18:01:03.997 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第1张票
2026-10-23 18:01:04.015 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第2张票
2026-10-23 18:01:04.016 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第3张票
2026-10-23 18:01:04.017 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第4张票
2026-10-23 18:01:04.018 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第5张票
2026-10-23 18:01:04.019 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第6张票
2026-10-23 18:01:04.019 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第7张票
2026-10-23 18:01:04.020 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第8张票
2026-10-23 18:01:04.020 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第9张票
2026-10-23 18:01:04.021 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor-12_0出售了第10张票
2026-10-23 18:01:04.021 | INFO     | __main__:sale_save:12 - ThreadPoolExecutor